# Phase 1 — Data Exploration Notebook
## Precision Agronomy Recommender | Pilot Crop: Paddy (Rice)

**Objective:** Load raw datasets, verify quality, understand distributions, identify missing values.
This notebook is the evidence base for the Phase 1 data summary report.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import json
import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12

print("✅ All imports successful")
print(f"   pandas  {pd.__version__}")
print(f"   numpy   {np.__version__}")

---
## Section 1: Crop Recommendation Dataset (NPK Baseline)
**Source:** Kaggle — Atharva Ingle / ICFA  
**File:** `data/raw/crop_recommendation.csv`

In [ ]:
# Load the dataset
df = pd.read_csv('../data/raw/crop_recommendation.csv')

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nCrops in dataset: {sorted(df['label'].unique())}")
print(f"\nTotal crops: {df['label'].nunique()}")

In [ ]:
# Filter to Paddy (Rice) only — our pilot crop
paddy_df = df[df['label'] == 'rice'].copy().reset_index(drop=True)

print(f"Total records: {len(df)}")
print(f"Paddy records: {len(paddy_df)} ({len(paddy_df)/len(df)*100:.1f}% of dataset)")
print()
print("── Paddy NPK statistics ──────────────────────────")
print(paddy_df[['N','P','K','ph','temperature','humidity','rainfall']].describe().round(2))

In [ ]:
# Check for missing values
print("Missing values in paddy dataset:")
missing = paddy_df.isnull().sum()
print(missing)
print()
if missing.sum() == 0:
    print("✅ No missing values in Kaggle dataset")
    print("⚠️  NOTE: This dataset lacks micronutrients (Zn, B, S, Fe, Mn, Cu)")
    print("   These will need KNN imputation using SHC data in Phase 2")
else:
    print(f"⚠️  {missing.sum()} missing values found — review before Phase 2")

In [ ]:
# NPK Distribution plots for Paddy
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

features = [('N', 'Nitrogen (kg/ha)', '#2E8B57'),
            ('P', 'Phosphorus (kg/ha)', '#4169E1'),
            ('K', 'Potassium (kg/ha)', '#DC143C')]

for ax, (col, label, color) in zip(axes, features):
    ax.hist(paddy_df[col], bins=20, color=color, alpha=0.75, edgecolor='white')
    ax.axvline(paddy_df[col].mean(), color='black', linestyle='--', linewidth=1.5, label=f'Mean: {paddy_df[col].mean():.1f}')
    ax.set_title(f'Paddy — {label}', fontweight='bold')
    ax.set_xlabel(label)
    ax.set_ylabel('Count')
    ax.legend(fontsize=10)

plt.suptitle('NPK Distributions for Paddy (Rice) — Kaggle Dataset', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/paddy_npk_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plot saved to data/processed/paddy_npk_distributions.png")

In [ ]:
# pH and rainfall distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(paddy_df['ph'], bins=15, color='#8B6914', alpha=0.75, edgecolor='white')
axes[0].axvline(5.5, color='green', linestyle='--', label='Optimal min (5.5)')
axes[0].axvline(7.0, color='red', linestyle='--', label='Optimal max (7.0)')
axes[0].set_title('Paddy — Soil pH Distribution', fontweight='bold')
axes[0].set_xlabel('pH')
axes[0].set_ylabel('Count')
axes[0].legend()

axes[1].hist(paddy_df['rainfall'], bins=15, color='#1E90FF', alpha=0.75, edgecolor='white')
axes[1].axvline(paddy_df['rainfall'].mean(), color='black', linestyle='--', label=f"Mean: {paddy_df['rainfall'].mean():.1f}mm")
axes[1].set_title('Paddy — Rainfall Distribution (mm)', fontweight='bold')
axes[1].set_xlabel('Rainfall (mm)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/processed/paddy_ph_rainfall.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Compare Paddy NPK vs all other crops — context for recommendation
crop_stats = df.groupby('label')[['N','P','K']].mean().sort_values('N', ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
x = range(len(crop_stats))
width = 0.25

bars_n = ax.bar([i - width for i in x], crop_stats['N'], width, label='N', color='#2E8B57', alpha=0.8)
bars_p = ax.bar(x, crop_stats['P'], width, label='P', color='#4169E1', alpha=0.8)
bars_k = ax.bar([i + width for i in x], crop_stats['K'], width, label='K', color='#DC143C', alpha=0.8)

# Highlight paddy
paddy_idx = list(crop_stats.index).index('rice')
for bar_group in [bars_n, bars_p, bars_k]:
    bar_group[paddy_idx].set_edgecolor('black')
    bar_group[paddy_idx].set_linewidth(2)

ax.set_xticks(x)
ax.set_xticklabels(crop_stats.index, rotation=45, ha='right')
ax.set_title('Mean NPK Requirements Across All Crops (Paddy highlighted)', fontweight='bold')
ax.set_ylabel('kg/ha')
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/all_crops_npk_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 2: Weather API Test — Open-Meteo
**Source:** Open-Meteo (no API key required)  
**Purpose:** Validate the 8-hour rain-free window logic for the spray decision engine

In [ ]:
# Test Open-Meteo API — Pilot district: Noida/Greater Noida, UP
PILOT_LAT = 28.4595
PILOT_LNG = 77.5022

url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude":  PILOT_LAT,
    "longitude": PILOT_LNG,
    "hourly":    "precipitation_probability,temperature_2m,relative_humidity_2m,wind_speed_10m",
    "forecast_days": 1,
    "timezone": "Asia/Kolkata"
}

response = requests.get(url, params=params)
print(f"API Status: {response.status_code}")

if response.status_code == 200:
    data = response.json()
    print(f"Location: {data.get('latitude')}, {data.get('longitude')}")
    print(f"Timezone: {data.get('timezone')}")
    print("✅ Open-Meteo API working")
else:
    print(f"❌ API Error: {response.text}")

In [ ]:
# Extract next 8 hours and run the spray window logic
hourly = data["hourly"]

# Next 8 hours
rain_8h    = hourly["precipitation_probability"][:8]
temps_8h   = hourly["temperature_2m"][:8]
humidity_8h= hourly["relative_humidity_2m"][:8]
times_8h   = hourly["time"][:8]

print("── Next 8 Hours: Rain Probability ────────────────")
for t, r in zip(times_8h, rain_8h):
    bar = "█" * (r // 10)
    flag = " ⚠️ HIGH" if r >= 20 else ""
    print(f"  {t[-5:]}  {r:3d}%  {bar}{flag}")

print()
max_rain = max(rain_8h)
safe_to_spray = max_rain < 20
print(f"Max rain probability in 8h: {max_rain}%")
print(f"Safe to spray?  {'✅ YES — GO ahead' if safe_to_spray else '❌ NO — DELAY spray'}")
print()
print(f"Avg temperature: {sum(temps_8h)/len(temps_8h):.1f}°C")
print(f"Avg humidity:    {sum(humidity_8h)/len(humidity_8h):.1f}%")

In [ ]:
# Visualize the 8-hour window
fig, ax = plt.subplots(figsize=(10, 4))

colors = ['#DC143C' if r >= 20 else '#2E8B57' for r in rain_8h]
bars = ax.bar(range(8), rain_8h, color=colors, alpha=0.8, edgecolor='white', linewidth=0.5)
ax.axhline(20, color='red', linestyle='--', linewidth=1.5, label='Delay threshold (20%)')
ax.set_xticks(range(8))
ax.set_xticklabels([t[-5:] for t in times_8h], rotation=45)
ax.set_xlabel('Hour (IST)')
ax.set_ylabel('Rain Probability (%)')
ax.set_title(f'8-Hour Rain Forecast — Noida, UP | {"🟢 SPRAY NOW" if safe_to_spray else "🔴 DELAY SPRAY"}', fontweight='bold')
ax.set_ylim(0, 100)
ax.legend()

plt.tight_layout()
plt.savefig('../data/processed/weather_spray_window.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ This chart is what drives the /weather-check API endpoint in Phase 4")

---
## Section 3: IFFCO Nano Urea Specs — Rules Verification
**Source:** `data/raw/iffco_nano_urea_specs.json`

In [ ]:
# Load and display IFFCO product rules
with open('../data/raw/iffco_nano_urea_specs.json') as f:
    specs = json.load(f)

nano_urea = specs['products']['nano_urea_plus']

print("── Nano Urea Plus — Key Specs ────────────────────")
print(f"  Nitrogen content : {nano_urea['specifications']['nitrogen_content']}")
print(f"  Particle size    : {nano_urea['specifications']['particle_size']}")
print(f"  Min dose         : {nano_urea['dosage']['min_ml_per_litre_water']} ml/L")
print(f"  Max dose         : {nano_urea['dosage']['max_ml_per_litre_water']} ml/L")
print()
print("── Paddy Application Windows ─────────────────────")
for w in nano_urea['application_windows']['paddy']:
    print(f"  Spray {w['spray_number']}: {w['stage']} — {w['timing']}")
print()
print("── Critical Rule ─────────────────────────────────")
print(f"  {nano_urea['constraints']['rule']}")
print()
print("── Environmental Impact ──────────────────────────")
ei = nano_urea['environmental_impact_vs_conventional_urea']
print(f"  N₂O reduction    : {ei['nitrous_oxide_reduction_pct']}%")
print(f"  NH₃ reduction    : {ei['ammonia_volatilization_reduction_pct']}%")

---
## Section 4: Phase 1 Data Summary

> Fill this in after running all cells above. This becomes your intern report Section 3.1.

In [ ]:
# Auto-generate Phase 1 summary stats for report
print("=" * 55)
print("  PHASE 1 DATA SUMMARY — for Intern Report")
print("=" * 55)
print()
print("DATASET 1: Kaggle Crop Recommendation")
print(f"  Total records       : {len(df)}")
print(f"  Paddy records       : {len(paddy_df)}")
print(f"  Features available  : {list(paddy_df.columns)}")
print(f"  Missing values      : {paddy_df.isnull().sum().sum()}")
print(f"  Missing features    : Zn, B, S, Fe, Mn, Cu (need KNN imputation)")
print()
print("DATASET 2: Weather API")
print(f"  Source              : Open-Meteo (dev), IMD (production)")
print(f"  Pilot location      : Noida, UP ({PILOT_LAT}, {PILOT_LNG})")
print(f"  Current spray safe? : {'YES' if safe_to_spray else 'NO'}")
print()
print("DATASET 3: IFFCO Rules Engine")
print(f"  Nano Urea dose range: 2.0 – 4.0 ml/L")
print(f"  Paddy spray windows : DAT 30–35 (tillering), DAT 45–55 (pre-flower)")
print(f"  Rain-free window    : 8 hours post-spray")
print()
print("NEXT STEPS (Phase 2):")
print("  1. Download SHC district data for Gautam Buddha Nagar")
print("  2. Download KCC pest query data (filter to Rice)")
print("  3. Build ETL pipeline to merge all sources")
print("  4. KNN imputation for missing micronutrients")
print("  5. GeoPandas spatial mapping of GPS → soil cluster")